# HDT 1: Pandas, SQL y DuckDB

**Ciencia de Datos, Sección A** · Asignada: martes 28 de julio · **Entrega: martes 4 de agosto, 23:59**

**Nombre:** Philip Falla - 20240667

Completar las celdas marcadas con `# ¿Qué va aquí?`. Cada ejercicio incluye una verificación comentada: descomentar para comprobar el resultado. Antes de entregar: **Kernel → Restart & Run All** (un notebook que no corre de arriba a abajo pierde 0.5 pts).

AI: resolver sin AI. Si se usó para entender un concepto, anotarlo en la mini-bitácora del final.

## Setup

Si falta DuckDB: descomentar la línea de instalación, ejecutar la celda una vez y volver a comentarla.

In [2]:
# !uv add duckdb    (en terminal)  o descomentar:  %pip install duckdb
import pandas as pd
import duckdb

URL = ("https://raw.githubusercontent.com/"
       "mwaskom/seaborn-data/master/penguins.csv")
penguins = pd.read_csv(URL)

# Tabla de nombres científicos (para los JOIN)
especies = pd.DataFrame({
    "species": ["Adelie", "Chinstrap", "Gentoo"],
    "nombre_cientifico": ["Pygoscelis adeliae",
                          "Pygoscelis antarcticus",
                          "Pygoscelis papua"],
})
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


## Parte A · Pandas (1.0 pt)

### Ejercicio 1 (0.10): cargar y explorar

Mostrar: (a) el `shape` del DataFrame, (b) los `dtypes`, y (c) cuántos nulos tiene **cada columna**.

In [9]:
print("Inciso a\n")
print(penguins.shape)
print("\nInciso b\n")
print(penguins.dtypes)
print("\nInciso c\n")
print(penguins.isna().sum())

# Verificación: el dataset original tiene 344 filas y 7 columnas,
# y la columna sex es la que más nulos tiene (11).

Inciso a

(344, 7)

Inciso b

species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object

Inciso c

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


### Ejercicio 2 (0.15): limpieza mínima

Crear un DataFrame `limpio` **sin** las filas que tengan nulo en cualquier columna. Reportar con un `print` cuántas filas se perdieron respecto al original.

In [ ]:
# mascara = penguins.isna()
# print(mascara)
# limpio = penguins[not mascara]
#
# limpio = penguins[(penguins["sex"] != None)]   # ¿Qué va aquí?
# limpio.head()

mascara = penguins.isna().any(axis=1)
limpio = penguins[~mascara].copy()

print(limpio)

filas_perdidas = penguins.isna().sum().sum()

print("Se perdieron " + str(filas_perdidas) + " filas con respecto al original")

# penguins.info()
# print(penguins.head)
# print(penguins[penguins["species"] != None].shape)
# limpio.isna().sum().sum()


# print(f"Se perdieron {...} filas")

# Verificación (descomentar):
assert limpio.shape[0] == 333 and limpio.isna().sum().sum() == 0

    species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen            39.1           18.7              181.0   
1    Adelie  Torgersen            39.5           17.4              186.0   
2    Adelie  Torgersen            40.3           18.0              195.0   
4    Adelie  Torgersen            36.7           19.3              193.0   
5    Adelie  Torgersen            39.3           20.6              190.0   
..      ...        ...             ...            ...                ...   
338  Gentoo     Biscoe            47.2           13.7              214.0   
340  Gentoo     Biscoe            46.8           14.3              215.0   
341  Gentoo     Biscoe            50.4           15.7              222.0   
342  Gentoo     Biscoe            45.2           14.8              212.0   
343  Gentoo     Biscoe            49.9           16.1              213.0   

     body_mass_g     sex  
0         3750.0    MALE  
1         3800.0  FEMALE  
2     

### Ejercicio 3 (0.15): máscaras y orden

De `limpio`: los pingüinos de la isla **Biscoe** con masa corporal **mayor a 4500 g**, ordenados de mayor a menor masa. Mostrar solo las columnas `species`, `island`, `body_mass_g`.

In [39]:
pesados_biscoe = limpio[(limpio["island"] == "Biscoe") & (limpio["body_mass_g"] > 4500)]   # ¿Qué va aquí?
pesados_biscoe = pesados_biscoe[["species", "island", "body_mass_g"]].sort_values(by="body_mass_g", ascending=False) ##

# Verificación (descomentar):
assert (pesados_biscoe["island"] == "Biscoe").all()
assert (pesados_biscoe["body_mass_g"] > 4500).all()
assert pesados_biscoe["body_mass_g"].is_monotonic_decreasing

### Ejercicio 4 (0.20): groupby con dos funciones

Masa corporal por **especie y sexo**: el **promedio** y el **conteo**, en una sola operación con `groupby` + `agg`.

In [40]:
# ¿Qué va aquí?

# limpio["media_peso"] = limpio.groupby("species")["body_mass_g"].transform("mean")
# limpio[["species", "sex", "media_peso"]].head(100)

resumen = limpio.groupby(["species", "sex"]).agg(
    peso_medio=("body_mass_g", "mean"),
    conteo=("species", "size"),
)

print(resumen.head(1000))
resumen.max()

# Verificación: el grupo más pesado debe ser Gentoo macho (~5485 g en promedio).

                   peso_medio  conteo
species   sex                        
Adelie    FEMALE  3368.835616      73
          MALE    4043.493151      73
Chinstrap FEMALE  3527.205882      34
          MALE    3938.970588      34
Gentoo    FEMALE  4679.741379      58
          MALE    5484.836066      61


peso_medio    5484.836066
conteo          73.000000
dtype: float64

### Ejercicio 5 (0.20): columna derivada

Agregar a `limpio` una columna `bill_ratio` = largo del pico / profundidad del pico. Mostrar el promedio de `bill_ratio` **por especie**, ordenado descendente. ¿Qué especie tiene el pico proporcionalmente más alargado?

In [ ]:
# ¿Qué va aquí?

limpio["bill_ratio"] = limpio["bill_length_mm"]/limpio["bill_depth_mm"]

# limpio["bill_mean"] = limpio.groupby("species")["bill_ratio"].transform("mean")
# limpio[["species", "bill_ratio", "bill_mean"]].head(100)

limpio.groupby("species").agg(
    bill_mean=("bill_ratio", "mean")
).head(1000).sort_values(by="bill_mean", ascending=False)


# Verificación: Gentoo debe quedar de primero (~3.2).

,bill_mean
species,
Gentoo,3.176602
Chinstrap,2.653756
Adelie,2.121478


### Ejercicio 6 (0.20): merge

Unir `limpio` con la tabla `especies` para que cada fila tenga su `nombre_cientifico`. Mostrar una fila de cada especie para comprobar.

In [55]:
# con_nombres = pd.merge(especies, penguins, on="species", how="right")

# ¿Qué va aquí?
con_nombres = pd.merge(especies, limpio, on="species", how="right")
con_nombres.drop_duplicates("species")[["species", "nombre_cientifico"]]

print(con_nombres.head(1000))

# Verificación (descomentar):
assert con_nombres.shape[0] == limpio.shape[0]
assert "nombre_cientifico" in con_nombres.columns

    species   nombre_cientifico     island  bill_length_mm  bill_depth_mm  \
0    Adelie  Pygoscelis adeliae  Torgersen            39.1           18.7   
1    Adelie  Pygoscelis adeliae  Torgersen            39.5           17.4   
2    Adelie  Pygoscelis adeliae  Torgersen            40.3           18.0   
3    Adelie  Pygoscelis adeliae  Torgersen            36.7           19.3   
4    Adelie  Pygoscelis adeliae  Torgersen            39.3           20.6   
..      ...                 ...        ...             ...            ...   
328  Gentoo    Pygoscelis papua     Biscoe            47.2           13.7   
329  Gentoo    Pygoscelis papua     Biscoe            46.8           14.3   
330  Gentoo    Pygoscelis papua     Biscoe            50.4           15.7   
331  Gentoo    Pygoscelis papua     Biscoe            45.2           14.8   
332  Gentoo    Pygoscelis papua     Biscoe            49.9           16.1   

     flipper_length_mm  body_mass_g     sex  bill_ratio  
0                

## Parte B · SQL con DuckDB (0.8 pt)

DuckDB consulta directamente los DataFrames en memoria: `duckdb.sql("SELECT ... FROM limpio")`. Cerrar cada consulta con `.df()` para ver el resultado como DataFrame.

### Ejercicio 7 (0.20): SELECT / WHERE / ORDER BY

El ejercicio 3, ahora en SQL: especie, isla y masa de los pingüinos de Biscoe con masa mayor a 4500 g, ordenados de mayor a menor.

In [46]:

# De `limpio`: los pingüinos de la isla Biscoe con masa corporal mayor a 4500 g, 
# ordenados de mayor a menor masa. Mostrar solo las columnas `species`, `island`, `body_mass_g`.


q7 = """
SELECT species, island, body_mass_g
FROM limpio
WHERE body_mass_g > 4500 AND island = 'Biscoe'
ORDER BY body_mass_g DESC
"""
duckdb.sql(q7).df()

# Verificación: debe dar las mismas filas que el ejercicio 3.

,species,island,body_mass_g
0,Gentoo,Biscoe,6300.0
1,Gentoo,Biscoe,6050.0
2,Gentoo,Biscoe,6000.0
3,Gentoo,Biscoe,6000.0
4,Gentoo,Biscoe,5950.0
...,...,...,...
101,Gentoo,Biscoe,4600.0
102,Gentoo,Biscoe,4600.0
103,Adelie,Biscoe,4600.0
104,Gentoo,Biscoe,4575.0


### Ejercicio 8 (0.20): GROUP BY + HAVING

Especies cuya masa corporal **promedio** supera los 4000 g, con su promedio redondeado.

In [ ]:
q8 = """
SELECT species, AVG(body_mass_g) AS mass_mean_g
FROM limpio
GROUP BY species
HAVING AVG(body_mass_g) > 4000
"""
duckdb.sql(q8).df()

# Verificación: solo una especie debe aparecer. ¿Cuál? Comparar con el resultado
# del ejercicio 4.

,species,mass_mean_g
0,Gentoo,5092.436975


### Ejercicio 9 (0.20): JOIN

El ejercicio 6, ahora en SQL: unir `limpio` con `especies` y mostrar especie, nombre científico y masa promedio por especie.

In [56]:
q9 = """
SELECT l.species, e.nombre_cientifico, AVG(l.body_mass_g) AS mean_mass_g
FROM limpio AS l
INNER JOIN especies AS e
    ON e.species = l.species
GROUP BY l.species, e.nombre_cientifico
"""
duckdb.sql(q9).df()

# Verificación: 3 filas, una por especie, cada una con su Pygoscelis.

,species,nombre_cientifico,mean_mass_g
0,Chinstrap,Pygoscelis antarcticus,3733.088235
1,Adelie,Pygoscelis adeliae,3706.164384
2,Gentoo,Pygoscelis papua,5092.436975


### Ejercicio 10 (0.20): window function

Los **3 pingüinos más pesados de cada especie**, usando `RANK() OVER (PARTITION BY ... ORDER BY ...)`. Pista de la sesión 3: la window function se calcula en una subconsulta y se filtra afuera.

In [65]:
q10 = """
SELECT species, body_mass_g,
           RANK() OVER (PARTITION BY species
                        ORDER BY body_mass_g DESC) AS rank_mass,
           AVG(body_mass_g) OVER (PARTITION BY species) AS media_peso_extra_info
    FROM limpio
    LIMIT 9
"""
duckdb.sql(q10).df()

# Verificación: alrededor de 9 filas (3 por especie; puede haber empates),
# y el rango nunca debe pasar de 3.

,species,body_mass_g,rank_mass,media_peso_extra_info
0,Gentoo,6300.0,1,5092.436975
1,Gentoo,6050.0,2,5092.436975
2,Gentoo,6000.0,3,5092.436975
3,Gentoo,6000.0,3,5092.436975
4,Gentoo,5950.0,5,5092.436975
5,Gentoo,5950.0,5,5092.436975
6,Gentoo,5850.0,7,5092.436975
7,Gentoo,5850.0,7,5092.436975
8,Gentoo,5850.0,7,5092.436975


## Parte C · Criterio (0.2 pt)

### Ejercicio 11 (0.20)

Los mismos análisis se resolvieron en Pandas y en SQL. En 3-4 líneas, **con base en el trabajo de esta hoja** (no de memoria): ¿cuándo conviene cada herramienta? Mencionar al menos una operación que resultó más natural en cada una.

**Respuesta:** Puramente con el trabajo de esta hoja, ambos son casi igual de buenos. A mi parecer era mucho mejor y más natural usar SQL en todo aspecto ya que es declarativo, se obtiene exactamente lo que declaras en el query. Por parte de pandas, me parece buena la parte del transform (que creo no usamos en este lab). El ejercicio 3 vs el 7: lo resolví mucho más rápido. Ya estoy acostumbrado a escribir queries y, además, usa lenguaje más natural. Pandas, en cambio, requiere conocer los métodos y atributos al ser nuevo para mi. El sort_values necesité buscar cómo se llamaba, por ejemplo.

De memoria: Es mejor SQL cuando tienes los datos en disco y pandas cuando es irrelevante cargar todo a memoria, o sea cuando no es muy grande la base de datos.

## Mini-bitácora de AI (opcional, no penaliza)

Si se usó AI para entender algún concepto, anotar aquí qué se preguntó y qué se entendió. Si no se usó, escribir "No se usó".

- Se utilizó para dudas de nombres de métodos, por ejemplo el sort values y axis en pandas.
- Se utilizó para recordar reglas de queries (ej. no se puede agrupar por agregados)
- Estuve un poco perdido con el tema del ejercicio 2 por que no encontré ningún ejemplo trabajado en clase. Se usó para tener la base para el resto de ejercicios. En los comentarios están los distintos intentos que se hicieron.
- Se evitó usar para temas como el del top 3 ranking por especie (ejercicio 10) por la integridad del lab o el print de filas del ejercicio 6. Otro ejemplo es el redondeo en SQL para el ejercicio 9.
- Me tardé 4 horas en terminar el lab con tal de usar solo las presentaciones y notebooks vistos en clase pero sí consideré necesario el uso de IA para lo previamente mencionado.
- Las líneas de código comentadas son como el 'procedimiento' de cómo llegué a mi respuesta

## Anexo: repaso de las sesiones 2 y 3

Regla de los ejercicios de NumPy: **sin ciclos `for`**.

In [67]:
import numpy as np

rng = np.random.default_rng(7)

### A1 (sesión 2): z-score sin loops

Normalizar un array: restar la media y dividir entre la desviación estándar.

In [68]:
alturas = rng.normal(170, 10, size=1000)

def z_score(x):
    return (x - x.mean()) / x.std()
    pass

z = z_score(alturas)
# Verificación (descomentar):
print(round(z.mean(), 4), round(z.std(), 4))  # ~0 y ~1

-0.0 1.0


### A2 (sesión 2): distancias con broadcasting

Distancia euclidiana de cada punto a un centro, sin loops.

In [69]:
puntos = rng.normal(size=(500, 2))   # 500 puntos en 2D
centro = np.array([1.0, 1.0])

# ¿Qué va aquí?
# Pista: (puntos - centro) usa broadcasting (500,2) - (2,)
# Luego: elevar al cuadrado, sumar con axis=1, sacar raíz
distancias = np.sqrt(np.sum((puntos - centro)**2, axis=1))

# ¿Cuántos puntos están a menos de 1 del centro?
cercanos = np.sum(distancias < 1)

# Verificación (descomentar):
print(distancias.shape)  # (500,)
print(cercanos)

(500,)
86


### A3 (sesión 3): propinas por día y turno

Dataset `tips` (propinas de un restaurante). `pct` = propina como fracción de la cuenta.

In [70]:
URL_TIPS = ("https://raw.githubusercontent.com/"
            "mwaskom/seaborn-data/master/tips.csv")
tips = pd.read_csv(URL_TIPS)
tips["pct"] = tips["tip"] / tips["total_bill"]
tips.head()

,total_bill,tip,sex,smoker,day,time,size,pct
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059447
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
3,23.68,3.31,Male,No,Sun,Dinner,2,0.139780
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808


In [71]:
# a) porcentaje medio de propina por dia
# b) por dia Y turno (day, time), en una tabla
# c) el dia con el mayor porcentaje medio
# d) numero de mesas por dia

resumen = tips.groupby("day").agg(
    pct_medio=("pct", "mean"),
    mesas=("total_bill", "count")
)

# Verificacion: resumen debe tener 4 filas
print(resumen)
print(resumen.shape)

      pct_medio  mesas
day                   
Fri    0.169913     19
Sat    0.153152     87
Sun    0.166897     76
Thur   0.161276     62
(4, 2)
